# Deepfake Audio Detection using Machine Learning & Deep Learning

This notebook demonstrates a complete end-to-end pipeline for classifying speech recordings as either **Genuine (Human)** or **Deepfake (AI-Generated)**.

## Objectives & Targets
- **Overall Accuracy**: &ge; 80%
- **Equal Error Rate (EER)**: &le; 12%
- **F1-Score**: &ge; 80%
- **Per-Class Accuracy**: &ge; 75%

## Methodology
1. **Dataset Generation**: We simulate genuine human voices (with natural pitch fluctuations and formants) and deepfake voices (with monotone pitch, phase distortions, and high-frequency buzz).
2. **Feature Extraction**: Extract Mel-Frequency Cepstral Coefficients (MFCCs), delta features, spectral centroid, spectral rolloff, and zero-crossing rate.
3. **Model Training**:
   - **Machine Learning**: Random Forest Classifier baseline.
   - **Deep Learning**: A 2D Convolutional Neural Network (CNN) trained on MFCC spectrogram maps.
4. **Evaluation**: Compute and plot accuracy, EER, confusion matrices, and per-class reports.

### Step 1: Import Dependencies

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

# Add the project root to path so we can import src modules
sys.path.append(os.path.abspath('..'))

### Step 2: Generate Simulated Dataset

We run the dataset preparation script to generate 100 genuine and 100 deepfake audio samples in `.wav` format.

In [ ]:
from scripts.prepare_dataset import main as generate_dataset
generate_dataset()

### Step 3: Feature Extraction

We extract:
- 1D statistical summary features for Random Forest.
- 2D MFCC spectrogram matrices for the PyTorch CNN.

In [ ]:
from src.features import process_file

base_dir = os.path.join("D:\\", "MARS_ML")
data_dir = os.path.join(base_dir, "data")
gen_dir = os.path.join(data_dir, "genuine")
fake_dir = os.path.join(data_dir, "deepfake")

genuine_files = [os.path.join(gen_dir, f) for f in os.listdir(gen_dir) if f.endswith('.wav')]
deepfake_files = [os.path.join(fake_dir, f) for f in os.listdir(fake_dir) if f.endswith('.wav')]

file_paths = genuine_files + deepfake_files
labels = [0]*len(genuine_files) + [1]*len(deepfake_files)

print(f"Extracting features for {len(file_paths)} files...")
feats_1d = []
feats_2d = []

for path in file_paths:
    f1d, f2d = process_file(path)
    feats_1d.append(f1d)
    feats_2d.append(f2d)
    
X_1d = np.array(feats_1d)
y = np.array(labels)
print(f"1D Feature matrix shape: {X_1d.shape}")
print(f"2D Feature matrix shape: ({len(feats_2d)}, {feats_2d[0].shape[0]}, {feats_2d[0].shape[1]})")

### Step 4: Visualizing Audio Features

Let's compare the waveform and spectrogram of a genuine voice sample and a deepfake voice sample.

In [ ]:
from src.features import load_audio
from scipy.signal import spectrogram

gen_y, sr = load_audio(genuine_files[0])
fake_y, _ = load_audio(deepfake_files[0])

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Genuine Waveform
axes[0, 0].plot(np.linspace(0, 2, len(gen_y)), gen_y, color='teal', alpha=0.8)
axes[0, 0].set_title("Genuine Audio Waveform")
axes[0, 0].set_ylabel("Amplitude")

# Deepfake Waveform
axes[0, 1].plot(np.linspace(0, 2, len(fake_y)), fake_y, color='crimson', alpha=0.8)
axes[0, 1].set_title("Deepfake Audio Waveform")

# Genuine Spectrogram
f, t, Sxx = spectrogram(gen_y, sr, nperseg=256, noverlap=128)
axes[1, 0].pcolormesh(t, f, 10 * np.log10(Sxx + 1e-10), cmap='viridis', shading='gouraud')
axes[1, 0].set_title("Genuine Spectrogram")
axes[1, 0].set_ylabel("Frequency (Hz)")
axes[1, 0].set_ylim(0, 8000)

# Deepfake Spectrogram
f, t, Sxx = spectrogram(fake_y, sr, nperseg=256, noverlap=128)
axes[1, 1].pcolormesh(t, f, 10 * np.log10(Sxx + 1e-10), cmap='viridis', shading='gouraud')
axes[1, 1].set_title("Deepfake Spectrogram")
axes[1, 1].set_ylim(0, 8000)

plt.tight_layout()
plt.show()

### Step 5: Train/Test Split

In [ ]:
X_train_1d, X_val_1d, train_paths, val_paths, train_labels, val_labels, train_idx, val_idx = train_test_split(
    X_1d, file_paths, y, np.arange(len(file_paths)), test_size=0.2, random_state=42, stratify=y
)

train_feats_2d = [feats_2d[idx] for idx in train_idx]
val_feats_2d = [feats_2d[idx] for idx in val_idx]

### Step 6: Train & Evaluate Random Forest Classifier

In [ ]:
from src.train import train_rf, evaluate_metrics

rf_model, rf_metrics = train_rf(X_train_1d, train_labels, X_val_1d, val_labels)

# Save Random Forest
os.makedirs(os.path.join(base_dir, "models"), exist_ok=True)
joblib.dump(rf_model, os.path.join(base_dir, "models", "rf_detector.pkl"))

### Step 7: Train & Evaluate PyTorch CNN

In [ ]:
from src.models import AudioDataset2D
from src.train import train_cnn

train_dataset = AudioDataset2D(train_paths, train_labels, train_feats_2d)
val_dataset = AudioDataset2D(val_paths, val_labels, val_feats_2d)

cnn_model, cnn_metrics = train_cnn(train_dataset, val_dataset, epochs=25, lr=0.001, device='cpu')

# Save CNN model weights
torch.save(cnn_model.state_dict(), os.path.join(base_dir, "models", "deepfake_detector.pth"))

### Step 8: Compare Models and Verify Targets

In [ ]:
print("Random Forest EER:", f"{rf_metrics['eer'] * 100:.2f}%")
print("PyTorch CNN EER:  ", f"{cnn_metrics['eer'] * 100:.2f}%")

# Check targets for CNN (or RF)
for name, metrics in [("Random Forest", rf_metrics), ("PyTorch CNN", cnn_metrics)]:
    print(f"\nVerifying {name} requirements:")
    assert metrics['accuracy'] >= 0.80, f"{name} failed overall accuracy target"
    assert metrics['eer'] <= 0.12, f"{name} failed Equal Error Rate (EER) target"
    assert metrics['f1'] >= 0.80, f"{name} failed F1-score target"
    assert metrics['acc_genuine'] >= 0.75, f"{name} failed Genuine class accuracy target"
    assert metrics['acc_deepfake'] >= 0.75, f"{name} failed Deepfake class accuracy target"
    print(f"  -> All targets successfully met for {name}!")